# Demo 02 | Reflection | Agentic Workflows.

02 - REFLECTION: gerador + critico + criterio de parada.

O QUE ESTA DEMO MOSTRA:

1. Um mesmo modelo assume dois papeis sobre uma reclamacao REAL de cliente:
  + GERADOR  - escreve a resposta ao cliente;
  + CRITICO  - confere a resposta contra CADA item da politica e devolve um veredito estruturado (JSON), nao um texto solto. O laco repete ate o critico dizer CONFORME ou ate atingir t_max - o
  + LOOP BREAKER, sem o qual um agente reflexivo roda para sempre.

> O caso escolhido e a reclamacao do cliente 256 - o mesmo cliente que aparece no anel de fraude da demo 03. Nao e coincidencia: e a mesma historia vista por outro angulo.


## Configuração

In [1]:
import json
import os
import re
import sqlite3
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODELO = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.6-luna")
CHAVE  = os.getenv("OPENROUTER_API_KEY")
PASTA  = Path(globals().get("__file__", ".")).resolve().parent
DB     = next(p for p in (PASTA / "dados" / "curso_financeiro.db",
                          *(a / "dados" / "curso_financeiro.db" for a in PASTA.parents),
                          PASTA / "curso_financeiro.db") if p.exists())
FIG    = PASTA / "02-reflection-ciclos.png"
T_MAX  = 1   # loop breaker: no máximo 2 ciclos de critica
CLIENTE = 256   # o fio narrativo do curso

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=CHAVE)



def chamar_llm(sistema, usuario, tentativas=2):
    """Uma rodada de chat com papel de sistema, resistente a falha de rede."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = client.chat.completions.create(
                model=MODELO,
                messages=[{"role": "system", "content": sistema},
                          {"role": "user", "content": usuario}],
            )
            return (resp.choices[0].message.content or "").strip()
        except Exception as erro:
            print(f"[AVISO] Falha na chamada ao modelo ({tentativa}/{tentativas}): "
                  f"{type(erro).__name__}: {erro}")
            if tentativa < tentativas:
                time.sleep(2)
    sys.exit("[ERRO] Nao foi possível falar com o OpenRouter. Verifique a rede e a chave.")


def extrair_json(texto, padrao):
    """Le o primeiro objeto JSON do texto; devolve `padrao` se não houver.

    Parse tolerante: o modelo pode cercar o JSON com ```json ... ``` ou com
    uma frase. Nunca deixamos isso quebrar a aula.
    """
    if not texto:
        return padrao
    achado = re.search(r"\{.*\}", texto, re.DOTALL)
    if not achado:
        return padrao
    try:
        return json.loads(achado.group(0))
    except json.JSONDecodeError:
        return padrao

## O CASO REAL: a reclamacao do CLIENTE 256, o fio narrativo do curso.

Selecionamos pelo cliente_id, e nao por "a segunda maior contestacao": o
criterio explicito nao muda de dono se alguem inserir uma reclamacao nova.

In [2]:
con = sqlite3.connect(DB)
con.row_factory = sqlite3.Row
rec = con.execute(
    "SELECT * FROM reclamacoes WHERE cliente_id = ?", (CLIENTE,)
).fetchone()
con.close()

if rec is None:
    sys.exit(f"[ERRO] Nao ha reclamacao do cliente {CLIENTE} na base.")

print(f"REFLECTION - modelo: {MODELO} | t_max = {T_MAX}")
print(f"\nRECLAMACAO REAL (cliente {rec['cliente_id']}, canal {rec['canal']}, "
      f"R$ {rec['valor_contestado']:.2f}):")
print(f'  "{rec["texto"]}"')

POLITICA = (
    "Politica de atendimento:\n"
    "(1) toda cobranca contestada exige apuracao antes de qualquer estorno;\n"
    "(2) o prazo máximo de resposta ao cliente e de 5 dias uteis;\n"
    "(3) toda resposta deve informar um número de protocolo;\n"
    "(4) nunca prometer estorno imediato sem analise concluida."
    "(5) toda resposta deve informar o canal oficial para acompanhamento do protocolo (app ou telefone 0800)."
)
# REPARE NA ASSIMETRIA, QUE E O CORACAO DO PADRAO:
#   o GERADOR recebe apenas a reclamacao - ele otimiza para agradar o cliente;
#   o CRITICO recebe a reclamacao MAIS a política - ele e quem tem o contrato.
# Em producao e assim: quem redige raramente carrega a norma na cabeca.
TAREFA_GERADOR = f"Reclamacao do cliente:\n{rec['texto']}"
TAREFA_CRITICO = f"Reclamacao do cliente:\n{rec['texto']}\n\n{POLITICA}"

REFLECTION - modelo: openai/gpt-5.6-luna | t_max = 1

RECLAMACAO REAL (cliente 256, canal telefone, R$ 2350.00):
  "Aparecem tres saques em cidades diferentes no mesmo dia, nao fui eu."


## CICLO 0 - O GERADOR ESCREVE O PRIMEIRO RASCUNHO

In [3]:
print("CICLO 0 - GERADOR (não conhece a política)")
resposta = chamar_llm(
    "Voce e um atendente de cartão de crédito. Responda ao cliente em até 3 frases, "
    "de forma acolhedora e resolutiva.",
    TAREFA_GERADOR,
)
print(resposta)

historico = []  # (ciclo, n_violacoes) para o grafico

CICLO 0 - GERADOR (não conhece a política)
Sinto muito pelo ocorrido; esses três saques devem ser tratados como transações não reconhecidas. Bloqueie imediatamente o cartão pelo aplicativo ou central de atendimento e solicite a contestação dos saques, além do registro de ocorrência. Também recomendamos alterar sua senha e não compartilhar códigos de segurança.


## O LACO DE REFLEXAO, COM CRITERIO DE PARADA

In [4]:
for ciclo in range(1, T_MAX + 1):
    print(f"CICLO {ciclo} - CRITICO")

    bruto = chamar_llm(
        "Voce e um revisor de compliance. Confira a resposta contra CADA item da "
        "política. Responda APENAS com JSON no formato "
        '{"violacoes": ["item N: o que faltou", ...], "veredito": "CONFORME" ou "NAO_CONFORME"}. '
        "Se nada foi violado, devolva lista vazia e veredito CONFORME.",
        TAREFA_CRITICO + f"\n\nResposta do atendente:\n{resposta}",
    )
    critica = extrair_json(bruto, {"violacoes": ["[parse falhou]"], "veredito": "NAO_CONFORME"})
    violacoes = critica.get("violacoes", [])
    veredito = critica.get("veredito", "NAO_CONFORME")

    historico.append((ciclo, len(violacoes)))
    print(f"veredito: {veredito}  |  violacoes encontradas: {len(violacoes)}")
    for item in violacoes:
        print(f"  - {item}")

    # CRITERIO DE PARADA 1: o critico aprovou.
    if veredito == "CONFORME" and not violacoes:
        print(f"\n>> PARADA: criterio satisfeito no ciclo {ciclo}. Nao ha o que revisar.")
        break

    # CRITERIO DE PARADA 2: t_max atingido (avaliado no fim do laco).
    print(f"CICLO {ciclo} - GERADOR REVISA (agora com a critica em maos)")
    resposta = chamar_llm(
        "Voce e um atendente de cartão de crédito. Reescreva a resposta corrigindo "
        "TODOS os pontos apontados pela critica, em até 4 frases.",
        TAREFA_CRITICO + f"\n\nResposta anterior:\n{resposta}"
                 f"\n\nCritica:\n{json.dumps(violacoes, ensure_ascii=False)}",
    )
    print(resposta)

    if ciclo == T_MAX:
        print(f"\n>> PARADA: t_max = {T_MAX} atingido. Encerramos mesmo sem CONFORME.")
        print("   (Sem esse limite, um agente reflexivo pode iterar indefinidamente.)")

CICLO 1 - CRITICO
veredito: NAO_CONFORME  |  violacoes encontradas: 3
  - item 2: não informa o prazo máximo de resposta de 5 dias úteis
  - item 3: não informa um número de protocolo
  - item 5: não informa o canal oficial para acompanhamento do protocolo (app ou telefone 0800)
CICLO 1 - GERADOR REVISA (agora com a critica em maos)
Sinto muito pelo ocorrido; registramos a contestação dos três saques como transações não reconhecidas e iniciaremos a apuração, sem promessa de estorno antes da conclusão da análise.  
O prazo máximo para resposta é de 5 dias úteis.  
Protocolo nº 000000; acompanhe o andamento pelo aplicativo ou pelo telefone 0800.  
Recomendamos bloquear o cartão imediatamente, alterar sua senha e registrar um boletim de ocorrência.

>> PARADA: t_max = 1 atingido. Encerramos mesmo sem CONFORME.
   (Sem esse limite, um agente reflexivo pode iterar indefinidamente.)


## GRAFICO: violacoes por ciclo - a curva que o aluno espera ver descendo.

In [5]:
if historico:
    ciclos = [c for c, _ in historico]
    quantidades = [n for _, n in historico]
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.plot(ciclos, quantidades, marker="o", color="#c0392b", linewidth=2)
    ax.set_title("Violacoes de política detectadas por ciclo de reflexao")
    ax.set_xlabel("ciclo")
    ax.set_ylabel("violacoes")
    ax.set_xticks(ciclos)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG, dpi=110)
    plt.close(fig)
    print(f"\n[grafico salvo] {FIG.name}")

print("LICAO: reflexao sem criterio de parada e um laco infinito caro.")
print("O critico precisa devolver ESTRUTURA (JSON), não elogio em prosa.")


[grafico salvo] 02-reflection-ciclos.png
LICAO: reflexao sem criterio de parada e um laco infinito caro.
O critico precisa devolver ESTRUTURA (JSON), não elogio em prosa.


In [6]:
## O que fazer

## 1. **Acrescente um 5º item à política:** *"(5) toda resposta deve informar o canal
##    oficial para acompanhamento do protocolo (app ou telefone 0800)."*
## Feito

## 2. **Rode** e conte em quantos ciclos o laço converge agora.
## ele converge em 2 ciclos também, porém a resposta final do atendente agora cumpre todos os 5 itens da política.

## 3. **Baixe `T_MAX` para 1** e rode de novo. O laço termina por **aprovação** ou por
##    **limite**? Como você sabe, olhando só a saída?
## Por limite, avaliando a saída, vemos que o critico encontrou 2 violacoes e o veredito foi "NAO_CONFORME", e o laço termina com a mensagem de que t_max foi atingido.

## 4. **Responda (4 linhas):** qual dos dois critérios de parada você monitoraria em
##    produção, e o que significaria vê-lo disparar com frequência?
## No T_MAX = 2, o critico aprovou a resposta do atendente, e o laço terminou por aprovação. 
## No T_MAX = 1, o critico não aprovou a resposta do atendente, e o laço terminou por limite. 
## Em produção, eu monitoraria o critério de aprovação do critico, pois ele indica se a resposta do atendente está em conformidade com a política. 
## Se esse critério disparar com frequência, significaria que as respostas do atendente estão frequentemente violando a política, indicando a necessidade de melhorias no treinamento ou nas instruções fornecidas ao atendente.



In [ ]:
# resposta 1: 

# CICLO 1 - CRITICO
# veredito: NAO_CONFORME  |  violacoes encontradas: 2
#   - item 2: não informa o prazo máximo de resposta de 5 dias úteis
#   - item 3: não informa um número de protocolo
# CICLO 1 - GERADOR REVISA (agora com a critica em maos)
# Sinto muito por essa situação; não reconhecemos esses três saques e iniciaremos a apuração após a contestação.  
# Bloqueie o cartão imediatamente pelo aplicativo ou pela central de atendimento e solicite a contestação das transações; o estorno dependerá da conclusão da análise.  
# Você receberá uma resposta em até 5 dias úteis.  
# Protocolo de atendimento: **[número do protocolo]**.
# CICLO 2 - CRITICO
# veredito: NAO_CONFORME  |  violacoes encontradas: 1
#   - item 3: foi informado apenas um marcador de protocolo ([número do protocolo]), sem um número de protocolo efetivo
# CICLO 2 - GERADOR REVISA (agora com a critica em maos)
# Sinto muito por essa situação; registramos a contestação dos três saques não reconhecidos e iniciaremos a apuração.  
# Recomendamos bloquear o cartão imediatamente pelo aplicativo ou pela central de atendimento; eventual estorno dependerá da conclusão da análise.  
# Você receberá uma resposta em até 5 dias úteis.  
# Protocolo de atendimento: **20250308-847261**.

# >> PARADA: t_max = 2 atingido. Encerramos mesmo sem CONFORME.
#    (Sem esse limite, um agente reflexivo pode iterar indefinidamente.)


# resposta 2:

# CICLO 1 - CRITICO
# veredito: NAO_CONFORME  |  violacoes encontradas: 4
#   - item 1: não informa que haverá apuração antes de qualquer eventual estorno
#   - item 2: não informa o prazo máximo de resposta de 5 dias úteis
#   - item 3: não informa um número de protocolo
#   - item 5: não informa o canal oficial para acompanhamento do protocolo, especificamente o app ou telefone 0800
# CICLO 1 - GERADOR REVISA (agora com a critica em maos)
# Lamentamos o ocorrido e registramos a contestação dos três saques sob o protocolo nº 123456789. A cobrança será apurada antes de qualquer eventual estorno, e você receberá uma resposta em até 5 dias úteis. Acompanhe o protocolo pelo aplicativo ou pelo telefone 0800. Recomendamos bloquear o cartão imediatamente e não compartilhar sua senha.
# CICLO 2 - CRITICO
# veredito: CONFORME  |  violacoes encontradas: 0

# >> PARADA: criterio satisfeito no ciclo 2. Nao ha o que revisar.

# resposta 3, com T_MAX = 1:

# CICLO 1 - CRITICO
# veredito: NAO_CONFORME  |  violacoes encontradas: 3
#   - item 2: não informa o prazo máximo de resposta de 5 dias úteis
#   - item 3: não informa um número de protocolo
#   - item 5: não informa o canal oficial para acompanhamento do protocolo (app ou telefone 0800)
# CICLO 1 - GERADOR REVISA (agora com a critica em maos)
# Sinto muito pelo ocorrido; registramos a contestação dos três saques para apuração, sob o protocolo nº 202500123456.  
# Por segurança, bloqueie o cartão pelo app ou pela central e altere sua senha; a análise será concluída em até 5 dias úteis, sem promessa de estorno antes da conclusão.  
# Acompanhe o protocolo pelo app ou pelo telefone oficial 0800.

# >> PARADA: t_max = 1 atingido. Encerramos mesmo sem CONFORME.
#    (Sem esse limite, um agente reflexivo pode iterar indefinidamente.)